### Multi-sheet integration status

#### Kept datasets
- `all_games`: main analysis dataset, one row per player-game
- `year_stats`: helper/reference dataset, one row per year
- `player_info`: helper lookup dataset, one row per player
- `countries`: helper lookup dataset, one row per country code

#### Rejected datasets
- `player_statistics_games`
- `validation`
- `all_time_leaderboard_stats`
- `yearly_leaderboard_stats`
- `custom_leaderboard_stats`

#### Join checks
- `all_games.player` → `player_info.Name`: safe  
  - unique players in `all_games`: 184
  - unmatched names in `all_games`: 0

- `player_info.Country Code` → `countries.Code`: safe  
  - unique country codes in `player_info`: 57
  - unmatched country codes: 0

#### Working decision
These datasets should stay as separate tables and be joined later in SQL/Tableau rather than merged now.

### Planned SQL table model

#### Main fact table
- `all_games`
  - grain: one row per player-game
  - primary analysis table

#### Lookup tables
- `player_info`
  - grain: one row per player
  - join key: `all_games.player = player_info.Name`

- `countries`
  - grain: one row per country code
  - join key: `player_info.Country Code = countries.Code`

#### Summary table
- `year_stats`
  - grain: one row per year
  - join key: `all_games.year = year_stats.Year`
  - use carefully because it is already aggregated

#### Modeling rule
Keep tables separate. Join only as needed based on the analysis grain.


### Sample merge test status

- A sample pandas merge was completed to validate the planned SQL join logic.
- `all_games` successfully joined to `player_info` on player/name.
- `player_info` successfully joined to `countries` on country code.
- Row count remained stable, so the joins do not create duplication.
- This merge is a validation step only, not the final data model.
- For the project, the tables should still remain separate and be joined later in SQL as needed.

In [1]:
import pandas as pd

In [3]:
all_games = pd.read_excel("../data/processed/ctwc_all_games_cleaned.xlsx")
year_stats = pd.read_excel("../data/processed/ctwc_year_stats_cleaned.xlsx")
player_info = pd.read_excel("../data/processed/ctwc_player_info_cleaned.xlsx")
countries = pd.read_excel("../data/processed/ctwc_countries_cleaned.xlsx")

In [4]:
all_games.shape, year_stats.shape, player_info.shape, countries.shape

((4202, 25), (14, 16), (892, 4), (213, 2))

In [6]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

In [7]:
all_games.head()

,year,round,match_id,game_id,game,player,won,playstyle,final_score,total_lines,no_mullen_score,no_mullen_lines,score_lvl_19_transition,lines_lvl_19_transition,score_lvl_29_transition,lines_lvl_29_transition,score_lvl_39_transition,lines_lvl_39_transition,topout_type,start_level,line_cap,same_piece_sets_active,post,post_post,game_link
0,2010,Finals,1,1,1,Jonas,Yes,DAS,530034,195.0,339042.0,135.0,522034.0,190.0,NaN,NaN,NaN,NaN,Natural,9,NaN,No,NaN,NaN,https://youtu.be/ZL4eRDOOP1I?t=24
1,2010,Finals,1,2,1,Harry Hong,No,DAS,302118,133.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Natural,9,NaN,No,NaN,NaN,https://youtu.be/ZL4eRDOOP1I?t=24
2,2010,Finals,1,3,2,Jonas,Yes,DAS,544534,245.0,542531.0,243.0,443999.0,192.0,NaN,NaN,NaN,NaN,Intentional,9,NaN,No,NaN,NaN,https://youtu.be/ZL4eRDOOP1I?t=595
3,2010,Finals,1,4,2,Harry Hong,No,DAS,517590,227.0,NaN,NaN,483111.0,191.0,NaN,NaN,NaN,NaN,Natural,9,NaN,No,NaN,NaN,https://youtu.be/ZL4eRDOOP1I?t=595
4,2011,Top 8,2,5,1,Jonas,Yes,DAS,164153,55.0,164153.0,55.0,NaN,NaN,NaN,NaN,NaN,NaN,Intentional,18,NaN,No,NaN,NaN,https://youtu.be/8sorxlk7QLs?t=14


In [9]:
player_info.head()

,Name,Twitch,YouTube,Country Code
0,Null,NaN,NaN,XAN
1,Player,NaN,NaN,XAN
2,-JJ,https://www.twitch.tv/terhoa,NaN,FIN
3,1stNoel,https://www.twitch.tv/the1stnoel,NaN,USA
4,20Dan03,https://www.twitch.tv/20dan03,https://www.youtube.com/@20Dan03,GBR


In [10]:
countries.head()

,Code,Name
0,AFG,Afghanistan
1,ALB,Albania
2,ALG,Algeria
3,AND,Andorra
4,ANG,Angola


In [11]:
year_stats.head()

,Year,Total Games,Median Score,Transitions,Killscreens,Level 39s,Transition %,Killscreen %,Level 39%,Average Score,Average Win,Average Loss,Median Lines,Average Lines,Median 19 Trans,Median 29 Trans
0,2025,356,1028918.5,325,242,11,0.912921,0.679775,0.030899,986860.0618,1.046003e+06,927717.4944,245.0,233.309659,544600.0,1031520.0
1,2024,354,1000380.0,318,217,8,0.898305,0.612994,0.022599,938261.6384,9.883792e+05,888144.0339,238.0,221.435028,545740.0,1032700.0
2,2023,354,928482.5,304,189,6,0.858757,0.533898,0.016949,848542.2797,8.955919e+05,801492.6441,231.0,205.014124,543010.5,1024623.0
3,2022,346,827740.5,279,140,9,0.806358,0.404624,0.026012,772039.3988,8.156813e+05,728397.4971,212.5,190.809249,543231.0,1023210.0
4,2021,984,754400.0,777,281,2,0.789634,0.285569,0.002033,709364.6982,7.625043e+05,656225.1098,192.0,175.374745,525220.0,1001400.0


In [12]:
all_games_players = set(all_games["player"].dropna().unique())
player_info_names = set(player_info["Name"].dropna().unique())

len(all_games_players), len(player_info_names)

(184, 892)

In [13]:
missing_in_player_info = sorted(all_games_players - player_info_names)
len(missing_in_player_info)

0

In [14]:
missing_in_player_info[:50]

[]

In [15]:
extra_in_player_info = sorted(player_info_names - all_games_players)
len(extra_in_player_info)

708

In [16]:
extra_in_player_info[:50]

['-JJ',
 '1stNoel',
 '20Dan03',
 '2DKaps',
 '321MrHaatz',
 '3RR0R404',
 '3arrett',
 '666_Tim',
 '8bitlord64',
 'AKGMB',
 'AMJ_Productions',
 'Aaron Lee',
 'AccessDEANied',
 'AdamIrish',
 'Adas',
 'Addison (CAN)',
 'Ajax',
 'AkAtu',
 'AkumaLuxray',
 'Al Shahed',
 'Alecat',
 'Alejodios41',
 'Alex (GBR)',
 'Alex (HKG)',
 'Alex S',
 'Alfalfa',
 'Alice P',
 'Allenbot',
 'Alwin',
 'Andrew Hunt',
 'Andrew L',
 'Andrew N',
 'Andrew4043',
 'Andrey (BRA)',
 'Andrey K',
 'Angel (HKG)',
 'Angelo',
 'Angrycoler',
 'Angus McDonald',
 'Anid29',
 'Anna D',
 'Anonwhyz',
 'Antonio',
 'Aoife',
 'AppleJuice',
 'Arattor',
 'Arbaro',
 'Archie Nash',
 'ArcticXC',
 'Ard']

In [19]:
player_info_codes = set(player_info["Country Code"].dropna().unique())
country_codes = set(countries["Code"].dropna().unique())

len(player_info_codes), len(country_codes)

(57, 213)

In [20]:
missing_country_codes = sorted(player_info_codes - country_codes)
len(missing_country_codes)

0

In [21]:
missing_country_codes

[]

## Merge test only

These temporary pandas merges were created only to validate the planned SQL join logic.

They are not final datasets and should not be exported or used as the main project tables. The final project model keeps the source tables separate and joins them later in SQL/Tableau as needed.

In [22]:
all_games_with_player_info = all_games.merge(
    player_info,
    left_on="player",
    right_on="Name",
    how="left",
    validate="many_to_one"
)

all_games_with_player_info.head()

,year,round,match_id,game_id,game,player,won,playstyle,final_score,total_lines,no_mullen_score,no_mullen_lines,score_lvl_19_transition,lines_lvl_19_transition,score_lvl_29_transition,lines_lvl_29_transition,score_lvl_39_transition,lines_lvl_39_transition,topout_type,start_level,line_cap,same_piece_sets_active,post,post_post,game_link,Name,Twitch,YouTube,Country Code
0,2010,Finals,1,1,1,Jonas,Yes,DAS,530034,195.0,339042.0,135.0,522034.0,190.0,NaN,NaN,NaN,NaN,Natural,9,NaN,No,NaN,NaN,https://youtu.be/ZL4eRDOOP1I?t=24,Jonas,https://www.twitch.tv/nubbinsgoody,https://www.youtube.com/@JonasTeh81,USA
1,2010,Finals,1,2,1,Harry Hong,No,DAS,302118,133.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Natural,9,NaN,No,NaN,NaN,https://youtu.be/ZL4eRDOOP1I?t=24,Harry Hong,https://www.twitch.tv/supasayajin,https://www.youtube.com/@SuPaSaYaJiN,USA
2,2010,Finals,1,3,2,Jonas,Yes,DAS,544534,245.0,542531.0,243.0,443999.0,192.0,NaN,NaN,NaN,NaN,Intentional,9,NaN,No,NaN,NaN,https://youtu.be/ZL4eRDOOP1I?t=595,Jonas,https://www.twitch.tv/nubbinsgoody,https://www.youtube.com/@JonasTeh81,USA
3,2010,Finals,1,4,2,Harry Hong,No,DAS,517590,227.0,NaN,NaN,483111.0,191.0,NaN,NaN,NaN,NaN,Natural,9,NaN,No,NaN,NaN,https://youtu.be/ZL4eRDOOP1I?t=595,Harry Hong,https://www.twitch.tv/supasayajin,https://www.youtube.com/@SuPaSaYaJiN,USA
4,2011,Top 8,2,5,1,Jonas,Yes,DAS,164153,55.0,164153.0,55.0,NaN,NaN,NaN,NaN,NaN,NaN,Intentional,18,NaN,No,NaN,NaN,https://youtu.be/8sorxlk7QLs?t=14,Jonas,https://www.twitch.tv/nubbinsgoody,https://www.youtube.com/@JonasTeh81,USA


In [23]:
all_games_with_player_info.shape

(4202, 29)

In [24]:
all_games_with_player_info[["player", "Name", "Country Code", "Twitch", "YouTube"]].head(10)

,player,Name,Country Code,Twitch,YouTube
0,Jonas,Jonas,USA,https://www.twitch.tv/nubbinsgoody,https://www.youtube.com/@JonasTeh81
1,Harry Hong,Harry Hong,USA,https://www.twitch.tv/supasayajin,https://www.youtube.com/@SuPaSaYaJiN
2,Jonas,Jonas,USA,https://www.twitch.tv/nubbinsgoody,https://www.youtube.com/@JonasTeh81
3,Harry Hong,Harry Hong,USA,https://www.twitch.tv/supasayajin,https://www.youtube.com/@SuPaSaYaJiN
4,Jonas,Jonas,USA,https://www.twitch.tv/nubbinsgoody,https://www.youtube.com/@JonasTeh81
5,Eli Markstrom,Eli Markstrom,USA,https://www.twitch.tv/elijahn5,https://www.youtube.com/@elijahn5
6,Jonas,Jonas,USA,https://www.twitch.tv/nubbinsgoody,https://www.youtube.com/@JonasTeh81
7,Eli Markstrom,Eli Markstrom,USA,https://www.twitch.tv/elijahn5,https://www.youtube.com/@elijahn5
8,Robin Mihara,Robin Mihara,USA,https://www.twitch.tv/tetrismattress,https://www.youtube.com/@robinmihara2418
9,Ben Mullen,Ben Mullen,USA,https://www.twitch.tv/benmullen297,https://www.youtube.com/@benmullen295


In [25]:
all_games_with_country = all_games_with_player_info.merge(
    countries,
    left_on="Country Code",
    right_on="Code",
    how="left",
    validate="many_to_one"
)

all_games_with_country.head()

,year,round,match_id,game_id,game,player,won,playstyle,final_score,total_lines,no_mullen_score,no_mullen_lines,score_lvl_19_transition,lines_lvl_19_transition,score_lvl_29_transition,lines_lvl_29_transition,score_lvl_39_transition,lines_lvl_39_transition,topout_type,start_level,line_cap,same_piece_sets_active,post,post_post,game_link,Name_x,Twitch,YouTube,Country Code,Code,Name_y
0,2010,Finals,1,1,1,Jonas,Yes,DAS,530034,195.0,339042.0,135.0,522034.0,190.0,NaN,NaN,NaN,NaN,Natural,9,NaN,No,NaN,NaN,https://youtu.be/ZL4eRDOOP1I?t=24,Jonas,https://www.twitch.tv/nubbinsgoody,https://www.youtube.com/@JonasTeh81,USA,USA,United States
1,2010,Finals,1,2,1,Harry Hong,No,DAS,302118,133.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Natural,9,NaN,No,NaN,NaN,https://youtu.be/ZL4eRDOOP1I?t=24,Harry Hong,https://www.twitch.tv/supasayajin,https://www.youtube.com/@SuPaSaYaJiN,USA,USA,United States
2,2010,Finals,1,3,2,Jonas,Yes,DAS,544534,245.0,542531.0,243.0,443999.0,192.0,NaN,NaN,NaN,NaN,Intentional,9,NaN,No,NaN,NaN,https://youtu.be/ZL4eRDOOP1I?t=595,Jonas,https://www.twitch.tv/nubbinsgoody,https://www.youtube.com/@JonasTeh81,USA,USA,United States
3,2010,Finals,1,4,2,Harry Hong,No,DAS,517590,227.0,NaN,NaN,483111.0,191.0,NaN,NaN,NaN,NaN,Natural,9,NaN,No,NaN,NaN,https://youtu.be/ZL4eRDOOP1I?t=595,Harry Hong,https://www.twitch.tv/supasayajin,https://www.youtube.com/@SuPaSaYaJiN,USA,USA,United States
4,2011,Top 8,2,5,1,Jonas,Yes,DAS,164153,55.0,164153.0,55.0,NaN,NaN,NaN,NaN,NaN,NaN,Intentional,18,NaN,No,NaN,NaN,https://youtu.be/8sorxlk7QLs?t=14,Jonas,https://www.twitch.tv/nubbinsgoody,https://www.youtube.com/@JonasTeh81,USA,USA,United States


In [26]:
all_games_with_country.shape

(4202, 31)

In [27]:
all_games_with_country[["player", "Country Code", "Name_y"]].head(10)

,player,Country Code,Name_y
0,Jonas,USA,United States
1,Harry Hong,USA,United States
2,Jonas,USA,United States
3,Harry Hong,USA,United States
4,Jonas,USA,United States
5,Eli Markstrom,USA,United States
6,Jonas,USA,United States
7,Eli Markstrom,USA,United States
8,Robin Mihara,USA,United States
9,Ben Mullen,USA,United States
